In [ ]:

#import libraries
!pip install openai pandas
from openai import OpenAI
import pandas as pd
import time

#setup
client = OpenAI(api_key="APIKEY")

MODEL = "gpt-4.1-mini"

#the instructions telling the AI to act like an Austrian Tax Expert
SYSTEM_PROMPT = """Du bist ein Experte für österreichisches Steuerrecht mit fundierten Kenntnissen in:
- EStG 1988 (Einkommensteuergesetz)
- UStG 1994 (Umsatzsteuergesetz)
- KStG (Körperschaftsteuergesetz)
- GrEStG 1987 (Grunderwerbsteuergesetz)

Beantworte Fragen auf Deutsch, präzise, fachlich korrekt und im Stil einer steuerrechtlichen Musterlösung.

Regeln:
- Antworte direkt ohne Einleitung.
- Verwende klare juristische Formulierungen.
- Zitiere immer die relevanten gesetzlichen Bestimmungen im Format: § [number] Abs. [x] [Gesetz Name].
- Führe Ausnahmen an, wenn diese bestehen.
- Vermeide Umgangssprache und Unsicherheiten.

Beispiel: "Das Einkommen... (§ 7 Abs. 1 KStG)"
"""

#load dataset
df = pd.read_csv("dataset_clean.csv", encoding="utf-8-sig") #store it in memory as 'df'
print(f"Total questions loaded: {len(df)}")

#function to get answer and error handling
#creates a reusable function that takes one question and returns one AI answer
def get_answer(question):
    try:
        response = client.chat.completions.create( #send the question and instructions to the OpenAI servers
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},#instructions how to answer
                {"role": "user", "content": question} #the actual tax question
            ],
            max_completion_tokens=200, #limit the answer length to save money(tokens)
            temperature=0 #greedy decoding
        )
        return response.choices[0].message.content.strip() #extract only the text of the answer from the large data packet received

    except Exception as e:
        print(f"Error: {e}") #if an error happens (like no internet), print the error message
        if "429" in str(e): #if we ask too fast (Error 429), the code waits for a full minute
            print("Rate limit! Waiting 60 seconds")
            time.sleep(60)
            try: #try the request one more time after the pause
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": question}
                    ],
                    max_completion_tokens=200,
                    temperature=0
                )
                return response.choices[0].message.content.strip()
            except Exception as e2: #if the second attempt fails, label the answer as "ERROR" in the file
                print(f"Second attempt failed: {e2}")
                return "ERROR"
        print("  Waiting 10 seconds") #for any other errors, wait 10 seconds and return "ERROR"
        time.sleep(10)
        return "ERROR"

#main loop
answers = []
ids = []

for i, row in df.iterrows(): #loop through every row in your table one by one
    question_id = row["id"] #get the unique ID for the current question
    question = row["prompt"] #get the actual question text from the 'prompt' column

    print(f"[{i+1}/643] Processing: {question_id}") #print he current progress, so you can monitor it

    answer = get_answer(question)#call our function to get the AI's response for this specific question
    #add the results to lists
    answers.append(answer)
    ids.append(question_id)

    print(f"Preview: {answer[:60]}...") #to see the results and check that everything is ok

    time.sleep(0.5) #wait 0.5 seconds between questions, not to overload the API server

    #every 50 questions, save a copy of the file in case the session crashes
    if (i + 1) % 50 == 0:
        temp_df = pd.DataFrame({"id": ids, "answer": answers})
        temp_df.to_csv("openai_results_temp.csv", index=False)
        print(f"\nProgress saved: {i+1} questions done\n")

#save final output
#combine the two lists into one final table
final_df = pd.DataFrame({
    "id": ids,
    "answer": answers
})

final_df.to_csv("openai_inference_results.csv", index=False)


print("DONE!")
print(f"Total answers: {len(final_df)}")
print(final_df.head(3))

Total questions loaded: 643
[1/643] Processing: CORP-TAX-001
  ✅ Preview: Die steuerliche Bemessungsgrundlage für die Körperschaftsteu...
[2/643] Processing: CORP-TAX-002
  ✅ Preview: Die Gewährung eines zinslosen Darlehens durch eine Körpersch...
[3/643] Processing: CORP-TAX-003
  ✅ Preview: Verpflichtet, sämtliche Einkünfte den Einkünften aus Gewerbe...
[4/643] Processing: CORP-TAX-004
  ✅ Preview: (a) Bei der österreichischen Tochtergesellschaft stellt die ...
[5/643] Processing: CORP-TAX-005
  ✅ Preview: Die steuerliche Behandlung einer offenen Ausschüttung unters...
[6/643] Processing: CORP-TAX-006
  ✅ Preview: Der maximal mögliche Verlustabzug im Veranlagungsjahr beträg...
[7/643] Processing: CORP-TAX-007
  ✅ Preview: Ein Mantelkauf liegt vor, wenn ein Unternehmen (meist eine K...
[8/643] Processing: CORP-TAX-008
  ✅ Preview: Ein Forderungsverzicht eines Gesellschafters gegenüber der K...
[9/643] Processing: CORP-TAX-009
  ✅ Preview: Die Verlustverrechnung innerhalb einer Körpers